In [1]:
import os
from pathlib import Path
import shutil

TEXBIN = "/Library/TeX/texbin"
if Path(TEXBIN).is_dir():
    os.environ["PATH"] = TEXBIN + ":" + os.environ.get("PATH", "")

print("pdflatex now:", shutil.which("pdflatex"))
print("latexmk now:", shutil.which("latexmk"))

pdflatex now: /Library/TeX/texbin/pdflatex
latexmk now: /Library/TeX/texbin/latexmk


In [2]:
# main.py
from pathlib import Path
from matplotlib.pyplot import title
import numpy as np
from openai import OpenAI

from config import get_experiment
from data_generation import generate_timeseries_dataset, stats_json
from variable_format import build_var_map_from_df_cols, remove_variables_from_first_paragraph, apply_variable_math
from llm_calls import llm_extended_body, llm_table_caption, llm_title
from tables import df_to_latex_summary_table, df_to_latex_full_table, insert_after_first_paragraph
from latex_build import build_tex_doc, compile_tex, show_first_latex_error
from llm_calls import llm_title

def main():
    experiment = get_experiment()

    client = OpenAI()
    rng = np.random.default_rng(experiment["rng_seed"])

    OUT = Path(experiment["out_dir"])
    OUT.mkdir(parents=True, exist_ok=True)

    for i in range(1, experiment["NUM_DATASETS"] + 1):
        print(f"▶ Dataset {i}/{experiment['NUM_DATASETS']}")

        df = generate_timeseries_dataset(experiment, rng)
        stats = stats_json(df)

        cols_for_varmap = [c for c in df.columns if c != "run_id"]
        VARMAP = build_var_map_from_df_cols(cols_for_varmap)
        VARS = sorted(VARMAP.keys(), key=len, reverse=True)

        body_raw = llm_extended_body(client, experiment, stats)
        body_raw = remove_variables_from_first_paragraph(body_raw, VARS)
        body = apply_variable_math(body_raw, VARMAP, VARS)

        cap1 = llm_table_caption(client, experiment, 1, df, table_kind="Summary statistics")
        table1_tex = df_to_latex_summary_table(df, cap1, label="tab:summary", VARMAP=VARMAP)
        body = insert_after_first_paragraph(body, table1_tex)

        if experiment.get("INCLUDE_FULL_TABLE", False):
            cap2 = llm_table_caption(client, experiment, 2, df, table_kind="Full recorded dataset")
            table2_tex = df_to_latex_full_table(df, cap2, label="tab:full", VARMAP=VARMAP, max_rows=None)
            body = body + "\n\n" + table2_tex


        title = llm_title(client, experiment, body, stats)
        tex_doc = build_tex_doc(title, body)
        assert tex_doc.count("{") == tex_doc.count("}"), "Brace mismatch before pdflatex"

        tex_out = OUT / f"paper{i:02d}.tex"
        tex_out.write_text(tex_doc, encoding="utf-8")

        ok, log = compile_tex(tex_out, OUT)
        if not ok:
            (OUT / f"paper{i:02d}.log.txt").write_text(log, encoding="utf-8")
            print(f"   ✖ compile failed; saved log to paper{i:02d}.log.txt")
            print("\n--- First LaTeX error snippet ---")
            print(show_first_latex_error(log))
            raise RuntimeError("pdflatex failed; inspect the snippet above and the saved log.")

        pdf_path = OUT / f"paper{i:02d}.pdf"
        if not pdf_path.exists():
            raise RuntimeError("Compilation succeeded but PDF not found where expected.")

        print("   ✔ saved", pdf_path)
        print("   ✔ saved", tex_out)

    print("🎉 Done. Files saved in:", OUT.resolve())

if __name__ == "__main__":
    main()

▶ Dataset 1/1
   ✔ saved Output/paper01.pdf
   ✔ saved Output/paper01.tex
🎉 Done. Files saved in: /Users/md_halim_mondol/Desktop/STEPS-LLM/steps-llm/Src/Modular Code/Output
